In [1]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

In [2]:
import numpy as np
import tifffile as tiff
import imageio.v3 as iio
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from core.data import CompressionMetrics
from core.utils import make_dual_update, round_sig
from core.video3d import reconstruct_video3d, compress_video3d

In [3]:
# Importing the video takes a while, so I put the import statement in its own cell.
# This way, we can make changes in the notebook without needing to import the video again.
video3d_raw = tiff.imread("../sources/Fluo-N3DL-DRO/01/*.tif")

In [34]:
## Drodophila full 3D video ##
# True dimensions are (50, 125, 603, 1272)
nz, ny, nx = 100, 100, 100
video3d = video3d_raw[:, :100, 200:300, 400:500]

In [35]:
# Hyperparameters
poly_degree = 6
t_degree = 35
lp_degree = 1.0
block_size = 10
cutoff = None

In [36]:
c_t, X_design, t_design_matrix, rescale = compress_video3d(
    video3d,
    poly_degree=poly_degree,
    t_degree=t_degree,
    block_size=block_size,
    lp_degree=lp_degree,
    cutoff=cutoff
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    3.1s
[Parallel(n_jobs=-1)]: Done 2320 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 98320 tasks      | elapsed:    8.0s
[Parallel(n_jobs=-1)]: Done 264208 tasks      | elapsed:   13.8s
[Parallel(n_jobs=-1)]: Done 466960 tasks      | elapsed:   20.9s
[Parallel(n_jobs=-1)]: Done 706576 tasks      | elapsed:   29.3s
[Parallel(n_jobs=-1)]: Done 800000 out of 800000 | elapsed:   32.7s finished


In [37]:
c_t = round_sig(c_t)
video3d_rec = reconstruct_video3d(
    video3d,
    block_size,
    c_t,
    X_design,
    t_design_matrix,
    rescale
)

In [38]:
# Saving coefficients
np.save("../results/video3d/drosophila_video_full_01/coefficients/coefficients__shape=%s__bs=%s__cutoff=%s__lp=%s__poly_deg=%s__t_degree=%s__dtype=%s.npy" %
(vid3d.shape, block_size, cutoff, lp_degree, poly_degree, t_degree, c_t.dtype), c_t, allow_pickle=False)

In [39]:
import napari

viewer = napari.Viewer()
viewer.add_image(video3d)
viewer.add_image(video3d_rec)
napari.run()

In [ ]:
metrics = CompressionMetrics(video3d, video3d_rec, c_t) 

In [ ]:
import pandas as pd
metrics_dict = {
    "Metric": ["Nonzero Coefficients (%)", "MSE", "PSNR (dB)", "Compression Ratio (%)", "Space Saved (%)", "SSIM"],
    "Value": [metrics.nz_percent, metrics.mse, metrics.psnr, metrics.compression_ratio, metrics.space_saved, metrics.ssim]
}
df = pd.DataFrame(metrics_dict)
print(df)